# Nova six-class ModernBERT router training
Ordered head labels: `call_safety`, `call_navigation`, `call_map`, `call_data`, `call_o2`, `llm`.
`call_llm`, `NO_TOOL`, and `OOD` merge into `llm`; `fine_label` is preserved. Frozen v3/v4 data are not used.
The output directory intentionally stays `models/binary`; this is NOT a binary head. Do not reuse an old binary/eight-class checkpoint.
Runtime > Change runtime type > **T4 GPU**. Before running, zip your local router on your machine:
```
cd /home/isaach/Nova && zip -qr colab/router.zip router -x 'router/.venv/*' '*/__pycache__/*' 'router/memory/*'
```

In [ ]:
!nvidia-smi -L
!pip -q install "transformers>=4.48" safetensors huggingface_hub numpy

In [ ]:
# Upload router.zip (the one made above)
from google.colab import files
import zipfile, os, shutil
up = files.upload()
shutil.rmtree('/content/Nova', ignore_errors=True)
os.makedirs('/content/Nova')
zipfile.ZipFile(next(iter(up))).extractall('/content/Nova')
%cd /content/Nova
!ls router | head -5; cat router/labels.json; wc -l router/data/*.jsonl

In [ ]:
# Download ModernBERT-base (add an HF token via Colab Secrets if rate limited)
from huggingface_hub import snapshot_download
BASE = snapshot_download('answerdotai/ModernBERT-base', local_dir='/content/models/ModernBERT-base')
print(BASE)
import json
assert json.load(open("router/labels.json"))["labels"] == ["call_safety", "call_navigation", "call_map", "call_data", "call_o2", "llm"]


In [ ]:
# Train six classes. Directory name stays binary for entrypoint compatibility; old heads are incompatible.
!rm -rf /content/models/binary
!python -m router.train --data-dir router/data --base /content/models/ModernBERT-base \
  --out-dir /content/models/binary --epochs 3 --batch-size 16 --eval-batch-size 32 \
  --device cuda --require-cuda --mixed-precision bf16

In [ ]:
# If the bf16 run errors on T4 (no bf16 support), rerun without --mixed-precision:
# !rm -rf /content/models/binary
# !python -m router.train --data-dir router/data --base /content/models/ModernBERT-base --out-dir /content/models/binary --epochs 3 --batch-size 16 --eval-batch-size 32 --device cuda --require-cuda

In [ ]:
# Package the checkpoint, save to Drive and download
!ls -la /content/models/binary
!cd /content/models && zip -qr binary.zip binary
from google.colab import drive, files
drive.mount('/content/drive')
!cp /content/models/binary.zip /content/drive/MyDrive/nova_router_binary.zip
files.download('/content/models/binary.zip')

Install locally:
```
mkdir -p /home/isaach/models/Nova/router && unzip binary.zip -d /home/isaach/models/Nova/router/
```
Then `/home/isaach/models/Nova/router/binary` is where Nova looks for the checkpoint.